# Stage 2: Data Preparation and Format Conversion

**Project:** Aspect-Level Customer Review Intelligence via Fine-Tuned Small Language Models

Converts the SemEval-2014 Laptops XML into instruction/JSON training pairs,
per `../docs/Customer_Review_Intelligence_Review0.md` §9, with the split
strategy revised in Appendix C (train/val/test all carved from
`Laptop_Train_v2.xml`, since the official test file has no gold polarity in
this mirror — see `notebooks/00_feasibility_and_eda.ipynb`).

**Inputs:** `data/raw/semeval/Laptop_Train_v2.xml`,
`data/raw/semeval/Laptops_Test_Data_phaseB.xml`.

**Outputs:** `data/processed/{train,val,test}.jsonl` (training pairs) and
`data/processed/phaseb_extraction_check.jsonl` (bonus, extraction-only
reference set — no polarity, per Decision D17).

**No model is called here.** This notebook only touches local files.


## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import model_config as mcfg
import semeval_parser as sp
import target_schema as ts

TRAIN_XML = ROOT / "data" / "raw" / "semeval" / "Laptop_Train_v2.xml"
PHASEB_XML = ROOT / "data" / "raw" / "semeval" / "Laptops_Test_Data_phaseB.xml"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42  # fixed and recorded per §20 reproducibility checklist

for p in [TRAIN_XML, PHASEB_XML]:
    print(f"{'OK ' if p.exists() else 'MISSING':8s} {p}")
print(f"\nActive prompt template source: src/model_config.py (model={mcfg.ACTIVE_MODEL})")


OK       /home/sreekanth/Downloads/asmta/data/raw/semeval/Laptop_Train_v2.xml
OK       /home/sreekanth/Downloads/asmta/data/raw/semeval/Laptops_Test_Data_phaseB.xml

Active prompt template source: src/model_config.py (model=qwen2.5:1.5b-instruct)


## 2. Parse & clean (§9.2 Decisions 1–2)

In [2]:
train_raw = sp.parse_semeval_xml(TRAIN_XML)
n_sentences_before = train_raw["sentence_id"].nunique()

train_clean = sp.drop_conflict_sentences(train_raw)
n_sentences_after = train_clean["sentence_id"].nunique()

print(f"Sentences before conflict-drop: {n_sentences_before}")
print(f"Sentences after conflict-drop:  {n_sentences_after}")
print(f"Dropped: {n_sentences_before - n_sentences_after}")

assert n_sentences_after == 3002, (
    f"Expected 3002 sentences after conflict-drop (per notebook 00's finding), "
    f"got {n_sentences_after} — data may have changed, investigate before proceeding."
)
print("\nMatches the 3,002-sentence count established in notebook 00. Proceeding.")


Sentences before conflict-drop: 3045
Sentences after conflict-drop:  3002
Dropped: 43

Matches the 3,002-sentence count established in notebook 00. Proceeding.


## 3. Build per-sentence records (target JSON + prompt)

In [3]:
def build_record(sentence_id: str, group: pd.DataFrame) -> dict:
    text = group["text"].iloc[0]
    aspect_rows = group[group["aspect"].notna()]
    aspects = list(zip(aspect_rows["aspect"], aspect_rows["polarity"]))
    target = ts.build_target(aspects)
    return {
        "sentence_id": sentence_id,
        "text": text,
        "prompt": mcfg.build_prompt(text),
        "target": target,
        # Compact, no whitespace — this is the literal completion string a
        # model is trained on, per §10.2 ("every space is a token").
        "completion": json.dumps(target, separators=(",", ":")),
    }

records = [
    build_record(sid, grp)
    for sid, grp in train_clean.groupby("sentence_id", sort=False)
]
print(f"Built {len(records)} records.")
records[0]


Built 3002 records.


{'sentence_id': '2339',
 'text': 'I charge it at night and skip taking the cord with me because of the good battery life.',
 'prompt': '### Instruction:\nExtract every product aspect mentioned in the following review sentence\nand the sentiment expressed towards each. Respond only with JSON in the\nschema: {"aspects":[{"aspect":"...","sentiment":"positive|negative|neutral"}]}\nIf no product aspect is mentioned, return {"aspects":[]}.\n\n### Input:\nI charge it at night and skip taking the cord with me because of the good battery life.\n\n### Response:\n',
 'target': {'aspects': [{'aspect': 'cord', 'sentiment': 'neutral'},
   {'aspect': 'battery life', 'sentiment': 'positive'}],
  'overall': 'positive'},
 'completion': '{"aspects":[{"aspect":"cord","sentiment":"neutral"},{"aspect":"battery life","sentiment":"positive"}],"overall":"positive"}'}

## 4. Sanity checks (§9.4)

In [4]:
# Every target round-trips as valid JSON.
bad_json = [r for r in records if json.loads(r["completion"]) != r["target"]]
print(f"Records failing JSON round-trip: {len(bad_json)}")
assert len(bad_json) == 0

# Every aspect string is a verbatim substring of its own sentence (§9.2 Decision 3).
bad_span = [
    (r["sentence_id"], a["aspect"])
    for r in records
    for a in r["target"]["aspects"]
    if a["aspect"] not in r["text"]
]
print(f"Aspect strings NOT found verbatim in their sentence: {len(bad_span)}")
if bad_span:
    print("  examples:", bad_span[:5])
assert len(bad_span) == 0

# Polarity values restricted to the three permitted strings.
bad_polarity = [
    (r["sentence_id"], a["sentiment"])
    for r in records
    for a in r["target"]["aspects"]
    if a["sentiment"] not in ts.ALLOWED_POLARITIES
]
print(f"Aspects with a disallowed polarity value: {len(bad_polarity)}")
assert len(bad_polarity) == 0

# Zero-aspect proportion — this is the empty-list baseline rate (§9.4).
n_zero_aspect = sum(1 for r in records if len(r["target"]["aspects"]) == 0)
print(f"\nZero-aspect records: {n_zero_aspect}/{len(records)} ({n_zero_aspect/len(records):.1%})")

print("\nAll sanity checks passed.")


Records failing JSON round-trip: 0
Aspect strings NOT found verbatim in their sentence: 0
Aspects with a disallowed polarity value: 0

Zero-aspect records: 1557/3002 (51.9%)

All sanity checks passed.


In [5]:
# Print ten random examples and actually read them (§9.4 — the instruction
# is explicit that this step is not optional).
rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(records), size=10, replace=False)
for i in sample_idx:
    r = records[i]
    print(f"[{r['sentence_id']}] {r['text']}")
    print(f"  -> {r['completion']}")
    print()


[198] do a quick search on the internet see if im the only one.
  -> {"aspects":[],"overall":"neutral"}

[2347] Being a PC user my whole life, it's taking a bit of time to adapt to the OS of a Mac but I'm finding my way around.
  -> {"aspects":[{"aspect":"OS","sentiment":"neutral"}],"overall":"neutral"}

[2616] I BOUGHT THEM FOR MY COLLEGE AGE GRANDCHILDREN AND THESE MACS ARE TOTALLY RELIABLE AND SUPPORT THEIR CLASS WORK 100% OF THE TIME.
  -> {"aspects":[],"overall":"neutral"}

[3056] The speed difference is next to NOTHING for a mac, and the hard drive can be manually upgraded or you could just buy a $60 500gb external hard drive.
  -> {"aspects":[{"aspect":"speed","sentiment":"negative"},{"aspect":"hard drive","sentiment":"neutral"},{"aspect":"500gb external hard drive","sentiment":"neutral"}],"overall":"negative"}

[589] I can't say enough good for it!
  -> {"aspects":[],"overall":"neutral"}

[938]   Now, as easy as it is to use, and I do think it is a great STARTER laptop.
  -> {"

## 5. Fixed-seed 80/10/10 split (§9.3, revised per Appendix C)

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
shuffled = rng.permutation(len(records))

n_total = len(shuffled)
n_val = round(n_total * 0.10)
n_test = round(n_total * 0.10)
n_train = n_total - n_val - n_test

val_idx = shuffled[:n_val]
test_idx = shuffled[n_val:n_val + n_test]
train_idx = shuffled[n_val + n_test:]

splits = {
    "train": [records[i] for i in train_idx],
    "val":   [records[i] for i in val_idx],
    "test":  [records[i] for i in test_idx],
}

for name, split_records in splits.items():
    n_zero = sum(1 for r in split_records if len(r["target"]["aspects"]) == 0)
    print(f"{name:5s}: {len(split_records):4d} sentences "
          f"({n_zero/len(split_records):.1%} zero-aspect)")

print(f"\nRandom seed: {RANDOM_SEED} (fixed and recorded)")


train: 2402 sentences (51.3% zero-aspect)
val  :  300 sentences (54.3% zero-aspect)
test :  300 sentences (54.0% zero-aspect)

Random seed: 42 (fixed and recorded)


Zero-aspect proportion should land close to the ~51% found on the full
pool in each split — a simple random shuffle, not a stratified one, so small
deviations are expected and fine; a large deviation would suggest a bug.


## 6. Write JSONL outputs

In [7]:
def write_jsonl(path: Path, split_records: list[dict]):
    with open(path, "w") as f:
        for r in split_records:
            f.write(json.dumps(r) + "\n")

for name, split_records in splits.items():
    out_path = PROCESSED_DIR / f"{name}.jsonl"
    write_jsonl(out_path, split_records)
    print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes, {len(split_records)} records)")


Wrote /home/sreekanth/Downloads/asmta/data/processed/train.jsonl (1,775,278 bytes, 2402 records)
Wrote /home/sreekanth/Downloads/asmta/data/processed/val.jsonl (221,659 bytes, 300 records)
Wrote /home/sreekanth/Downloads/asmta/data/processed/test.jsonl (222,815 bytes, 300 records)


## 7. Bonus extraction-only check set — `phaseB` (Decision D17)

`Laptops_Test_Data_phaseB.xml` has aspect-term spans but no gold polarity
(confirmed in notebook 00). It has ~zero text overlap with the training pool,
so even without polarity it's a genuinely independent, larger-scale sanity
check for aspect *extraction* (M2) alone — not training data, not part of
the train/val/test split above, and never scored on polarity/Joint-F1.


In [8]:
phaseb_raw = sp.parse_semeval_xml(PHASEB_XML)
# No conflict-drop needed: this file carries no polarity at all, so there is
# no `conflict` label to find.

def build_extraction_record(sentence_id: str, group: pd.DataFrame) -> dict:
    text = group["text"].iloc[0]
    aspect_rows = group[group["aspect"].notna()]
    gold_aspects = sorted(aspect_rows["aspect"].tolist())
    return {
        "sentence_id": sentence_id,
        "text": text,
        "prompt": mcfg.build_prompt(text),
        "gold_aspects": gold_aspects,  # extraction-only gold reference, no polarity
    }

phaseb_records = [
    build_extraction_record(sid, grp)
    for sid, grp in phaseb_raw.groupby("sentence_id", sort=False)
]

out_path = PROCESSED_DIR / "phaseb_extraction_check.jsonl"
write_jsonl(out_path, phaseb_records)
print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes, {len(phaseb_records)} records)")


Wrote /home/sreekanth/Downloads/asmta/data/processed/phaseb_extraction_check.jsonl (452,049 bytes, 800 records)


## 8. Manifest

In [9]:
manifest = {
    "random_seed": RANDOM_SEED,
    "source_files": {
        "train_val_test": str(TRAIN_XML.relative_to(ROOT)),
        "extraction_check": str(PHASEB_XML.relative_to(ROOT)),
    },
    "conflict_sentences_dropped": n_sentences_before - n_sentences_after,
    "splits": {name: len(r) for name, r in splits.items()},
    "extraction_check_sentences": len(phaseb_records),
    "prompt_template_source": "src/model_config.py:PROMPT_TEMPLATE",
    "target_schema_source": "src/target_schema.py:build_target",
}

manifest_path = PROCESSED_DIR / "MANIFEST.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {manifest_path}\n")
print(json.dumps(manifest, indent=2))


Wrote /home/sreekanth/Downloads/asmta/data/processed/MANIFEST.json

{
  "random_seed": 42,
  "source_files": {
    "train_val_test": "data/raw/semeval/Laptop_Train_v2.xml",
    "extraction_check": "data/raw/semeval/Laptops_Test_Data_phaseB.xml"
  },
  "conflict_sentences_dropped": 43,
  "splits": {
    "train": 2402,
    "val": 300,
    "test": 300
  },
  "extraction_check_sentences": 800,
  "prompt_template_source": "src/model_config.py:PROMPT_TEMPLATE",
  "target_schema_source": "src/target_schema.py:build_target"
}


### Summary

- 3,002 non-conflict sentences parsed and converted, split 80/10/10 with a
  fixed, recorded seed (42) into `data/processed/{train,val,test}.jsonl`.
- Every target JSON-validated, every aspect span verified verbatim in its
  source sentence, every polarity value restricted to the allowed set —
  all sanity checks from §9.4 pass.
- `data/processed/phaseb_extraction_check.jsonl` holds 800 independent
  sentences for a bonus extraction-only (M2) check later — no polarity, not
  used for training, not part of the split above (Decision D17).
- **Next step:** Stage 5 zero-shot baseline (`notebooks/02_baseline.ipynb`) —
  the first point this project actually calls the Ollama model.
